# Resume training the 23.4M OrbitNet BC from epoch-2 (A100)

Continues the Stage-B 23.4M gate-head BC (`640/L6/ff1280/h8`) **from the epoch-2 checkpoint**,
saving a checkpoint **every epoch** to Drive (both the rolling `.last.pt` and a per-epoch
archive `ep_NNN.pt`).

**Why resume:** the 23.4M was *undertrained* — its arena win-rate vs producer climbed every
epoch (17.5% → 33% → 40%) and by epoch 2 nearly matched the 15M's peak (45%), still rising.
More epochs should push it past the 15M.

**⚠️ Data note:** the original 26M-state Lambda shards are gone (instance terminated). This
resumes on the **12.8M-state shards on Drive** (`MyDrive/orbit_wars/shards`) — same top-ladder
win-weighted BC regime, just a smaller corpus. The model + optimizer continue fine. To train on
a bigger corpus instead, first rebuild shards with the harvest cells in `winbc_colab.ipynb`,
then point `SHARDS` below at them.

**⚠️ Gate by ARENA, not val-`sel`:** best-on-`sel` diverged from arena and the arena win-rate
OSCILLATES across epochs (the 15M went 28→45→8→37). So this notebook archives **every** epoch
(`--archive-dir`); pull each `ep_NNN.pt` and arena-gate it (`scripts/arena.py`, executor
`min`/`0.5`) to find the peak — don't trust best-on-`sel` or the last epoch.

**Disconnect-proof:** every cell writes to Drive. If Colab drops, just re-run all cells — the
train cell resumes from the latest `.last.pt` on Drive automatically.


In [ ]:
# === Cell 1: Setup — mount Drive, clone/pull repo, install, GPU check ===
import os, sys
from google.colab import drive
drive.mount('/content/drive')

DRIVE  = '/content/drive/MyDrive/orbit_wars'
SHARDS = f'{DRIVE}/shards'          # 12.8M-state shards from the earlier run (training data)
CKPTS  = f'{DRIVE}/checkpoints'     # checkpoints + curves persist here (durable)
for d in (DRIVE, SHARDS, CKPTS):
    os.makedirs(d, exist_ok=True)

REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'   # <-- EDIT if needed
REPO_DIR = '/content/orbit_wars'
if os.path.exists(REPO_DIR):
    os.system(f'cd {REPO_DIR} && git pull')   # pull gets the --archive-dir flag + this notebook
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
os.system('pip install -q torch numpy pyyaml')   # training needs no kaggle-environments
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('CUDA:', torch.cuda.is_available(), name)
assert 'A100' in name or torch.cuda.is_available(), 'Pick a GPU runtime (A100 recommended)'
os.system('df -h /content | tail -1')


In [ ]:
# === Cell 2: Stage shards to fast local SSD ===
# Training re-reads shards each epoch; the Drive MOUNT is ~20x slower than /content SSD, so
# copy them local once (RAM-cache then keeps them decoded in memory).
import os, json
SHARDS_LOCAL = '/content/shards'
os.makedirs(SHARDS_LOCAL, exist_ok=True)
if os.path.exists(f'{SHARDS_LOCAL}/manifest.json'):
    print('shards already staged on local /content')
elif os.path.exists(f'{SHARDS}/manifest.json'):
    print('staging Drive shards -> local /content (one-time copy)...')
    os.system(f'cp -r {SHARDS}/* {SHARDS_LOCAL}/')
    print('done')
else:
    raise SystemExit(f'No shards at {SHARDS}. Build them first (winbc_colab.ipynb harvest cells).')
m = json.load(open(f'{SHARDS_LOCAL}/manifest.json'))
print(f"shards: {m['n_examples']:,} examples, {m['n_shards']} files, w_loss={m['w_loss']}")


In [ ]:
# === Cell 3: Seed the resume from the epoch-2 checkpoint (first run only) ===
# winbc_train --resume loads <out>.last.pt. On the FIRST run we seed that with the epoch-2
# checkpoint so training continues from epoch 3. On a re-run after a disconnect, <out>.last.pt
# is already the latest epoch -> we DON'T overwrite it (so we don't lose progress).
import os, shutil, torch
OUT     = f'{CKPTS}/winbc_24m_resume.pt'
LAST    = f'{CKPTS}/winbc_24m_resume.last.pt'
CURVES  = f'{CKPTS}/winbc_24m_resume_curves.csv'
ARCHDIR = f'{CKPTS}/winbc_24m_resume_epochs'   # per-epoch ep_NNN.pt land here (arena-gate each)
SEED    = f'{CKPTS}/winbc_24m_ep2.pt'          # the epoch-2 checkpoint (already on Drive)

if os.path.exists(LAST):
    ep = torch.load(LAST, map_location='cpu', weights_only=False).get('epoch')
    print(f'Found existing {os.path.basename(LAST)} (epoch {ep}) -> RESUMING from there (disconnect recovery). Not re-seeding.')
elif os.path.exists(SEED):
    shutil.copy(SEED, LAST)
    ck = torch.load(LAST, map_location='cpu', weights_only=False)
    print(f"Seeded resume from epoch-2 (epoch={ck.get('epoch')}, best sel={ck.get('best'):.3f}, arch={ck.get('arch')}).")
    print(f'Training will CONTINUE from epoch {int(ck.get("epoch"))+1}.')
else:
    raise SystemExit(f'Missing both {LAST} and {SEED}. Upload winbc_24m_ep2.pt to {CKPTS}/ first.')


In [ ]:
# === Cell 4: Train (resume the 23.4M; save EVERY epoch to Drive) ===
# Arch MUST match the checkpoint: 640/L6/ff1280/h8. Batch 512 fits A100-40GB (~20GB); the
# loaded optimizer state restores the original LR (~4e-4), so we just continue the trajectory.
# Saves each epoch: best-on-sel -> {OUT}, rolling .last.pt -> {LAST}, per-epoch -> {ARCHDIR}/ep_NNN.pt
# Re-run this cell after any disconnect; it picks up from the latest .last.pt on Drive.
EPOCHS = 20   # ceiling; with per-epoch archiving you can stop & arena-gate anytime
BATCH  = 512  # reduce to 384/256 if you get a non-40GB GPU and it OOMs
!python scripts/winbc_train.py --shards {SHARDS_LOCAL} --gate_head --resume \
    --embed-dim 640 --n-layers 6 --ff-dim 1280 --n-heads 8 \
    --epochs {EPOCHS} --batch {BATCH} --lr 4e-4 --launch_weight 8.0 --w_loss 0.3 \
    --shard-cache auto \
    --out {OUT} --curves {CURVES} --archive-dir {ARCHDIR}


## After training — find the peak epoch (arena-gate each)

Per-epoch checkpoints are at `MyDrive/orbit_wars/checkpoints/winbc_24m_resume_epochs/ep_NNN.pt`.
Because arena win-rate oscillates and diverges from val-`sel`, **gate each one** to pick the best:

```bash
# locally (or on a box), per epoch checkpoint:
OW_BC_TEACHER_CKPT=ep_003.pt OW_BC_DRAIN=min OW_BC_GATE_THR=0.5 \
  uv run python scripts/arena.py --agents bc_teacher,producer,v5 --games 120 --workers N
```

Keep the epoch with the **highest win-rate vs producer** (not the last, not best-on-`sel`).
Then build a submission bundle from it:
`uv run python scripts/build_bc_bundle.py --ckpt ep_NNN.pt --drain min --gate-thr 0.5`.

**Curves** (`winbc_24m_resume_curves.csv`) track val metrics per epoch, but remember these are
only weakly correlated with arena — trust the arena gate.
